<a href="https://colab.research.google.com/github/mmedikon/rag_on_vLLM_Qwen/blob/main/rag_on_vLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install vllm

  Using cached vllm-0.29.0-cp38-abi3-manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached torchaudio-2.11.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached vllm-0.29.0-cp38-abi3-manylinux_2_28_x86_64.whl (316.0 MB)
Using cached torchaudio-2.11.0-cp313-cp313-manylinux_2_28_x86_64.whl (1.8 MB)


In [ ]:
!vllm --version

0.29.0


In [ ]:
!pip install sentence-transformers faiss-cpu

  Using cached faiss_cpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 20.2 MB/s eta 0:00:00
Using cached faiss_cpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (18.8 MB)


In [ ]:
import subprocess
import time

# Start vLLM OpenAI-compatible server in the background
# We use a lightweight Qwen 1.5B model which is fast and fits into Colab's free T4 GPU
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

vllm_process = subprocess.Popen([
    "vllm", "serve", model_name,
    "--port", "8001",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.8"  # Leave some VRAM for embedding models if needed
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print("Starting vLLM server... This takes about 1-2 minutes to download weights and initialize.")
# Wait for the server to spin up and bind to port 8000
time.sleep(90)
print("vLLM Server should be ready now!")

Starting vLLM server... This takes about 1-2 minutes to download weights and initialize.
vLLM Server should be ready now!


In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.9 MB/s eta 0:00:00


In [ ]:
!pip install langchain-text-splitters colab-xterm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 5.7 MB/s eta 0:00:00


In [ ]:

import os
import sys

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from openai import OpenAI

from documents import DOCUMENTS

load_dotenv()  # reads OPENROUTER_API_KEY etc. from a local .env file

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

CHROMA_PERSIST_DIR = "./chroma_store"
COLLECTION_NAME = "rag_demo"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"  # small, fast, runs locally

OPENROUTER_BASE_URL = "http://localhost:8001/v1"
OPENROUTER_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # Qwen-Plus, served through OpenRouter
os.environ["API_KEY"] ="EMPTY"

TOP_K = 3  # how many chunks to retrieve per query

SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer the user's question using ONLY the "
    "provided context. If the context doesn't contain the answer, say you "
    "don't know instead of making something up. Cite which source(s) you "
    "used by their 'source' name."
)


# ---------------------------------------------------------------------------
# Vector store (ChromaDB)
# ---------------------------------------------------------------------------

def get_chroma_collection():
    """Create (or load) a persistent ChromaDB collection with a local
    sentence-transformers embedding function attached to it."""

    client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

    embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name=EMBEDDING_MODEL_NAME
    )

    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=embed_fn,
        metadata={"hnsw:space": "cosine"},
    )
    return collection


def ingest_documents(collection):
    """Add the sample documents to the collection if it's empty.
    ChromaDB computes the embeddings automatically via the collection's
    embedding function."""

    if collection.count() > 0:
        print(f"[ingest] Collection already has {collection.count()} docs, skipping ingest.")
        return

    collection.add(
        ids=[d["id"] for d in DOCUMENTS],
        documents=[d["text"] for d in DOCUMENTS],
        metadatas=[d["metadata"] for d in DOCUMENTS],
    )
    print(f"[ingest] Added {len(DOCUMENTS)} documents to '{COLLECTION_NAME}'.")


def retrieve(collection, query: str, top_k: int = TOP_K):
    """Run a similarity search and return the matching chunks + metadata."""

    results = collection.query(query_texts=[query], n_results=top_k)

    chunks = []
    for text, meta, distance in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        chunks.append({"text": text, "metadata": meta, "distance": distance})
    return chunks


# ---------------------------------------------------------------------------
# Generation (Qwen-Plus via OpenRouter)
# ---------------------------------------------------------------------------

def get_openrouter_client() -> OpenAI:
    api_key = os.environ.get("API_KEY")
    if not api_key:
        raise RuntimeError(
            "OPENROUTER_API_KEY is not set. Copy .env.example to .env and add your key, "
            "or `export OPENROUTER_API_KEY=...` before running."
        )

    return OpenAI(
        base_url=OPENROUTER_BASE_URL,
        api_key=api_key,

    )


def build_context_block(chunks) -> str:
    lines = []
    for i, c in enumerate(chunks, start=1):
        source = c["metadata"].get("source", "unknown")
        lines.append(f"[{i}] (source: {source})\n{c['text']}")
    return "\n\n".join(lines)


def generate_answer(client: OpenAI, query: str, chunks) -> str:
    context_block = build_context_block(chunks)

    user_prompt = (
        f"Context:\n{context_block}\n\n"
        f"Question: {query}\n\n"
        "Answer using only the context above."
    )

    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


# ---------------------------------------------------------------------------
# End-to-end RAG call
# ---------------------------------------------------------------------------

def answer_question(collection, client: OpenAI, query: str, verbose: bool = True):
    chunks = retrieve(collection, query)

    if verbose:
        print("\n--- Retrieved context ---")
        for c in chunks:
            print(f"  (dist={c['distance']:.4f}, source={c['metadata'].get('source')}) "
                  f"{c['text'][:90]}...")
        print("--------------------------\n")

    answer = generate_answer(client, query, chunks)
    return answer


# ---------------------------------------------------------------------------
# CLI entry point
# ---------------------------------------------------------------------------

def main():
    collection = get_chroma_collection()
    ingest_documents(collection)
    client = get_openrouter_client()
    query = "Why chroma DB used for? "
    answer = answer_question(collection, client, query)
    print(f"Q: {query}\n\nA: {answer}")



if __name__ == "__main__":
    main()



[ingest] Collection already has 6 docs, skipping ingest.

--- Retrieved context ---
  (dist=0.4399, source=chroma_overview) ChromaDB is an open-source embedding database designed for building AI applications with v...
  (dist=0.6483, source=rag_pipeline) In a typical RAG pipeline, documents are first split into chunks, then converted into vect...
  (dist=0.7660, source=qwen_overview) Qwen-Plus is a large language model in Alibaba's Qwen series, positioned as a balanced opt...
--------------------------

Q: Why chroma DB used for? 

A: ChromaDB is used for building AI applications with vector search, supporting the storage of documents along with their embeddings and metadata, running in memory, persisting to disk, or as a client-server deployment.


In [ ]:
vllm_process.terminate()